# NB_lab_test_trends

Builds `adb_caresync.gold.lab_test_trends` (one row per test type per calendar month) from silver lab_results and calendar, with a month-over-month LAG trend on abnormal rate, per the STTM `sttm_silver_to_gold.xlsx`. Full rebuild (CREATE OR REPLACE).

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.lab_test_trends
COMMENT 'Gold: one row per test type per calendar month with abnormal/critical rates and month-over-month trend (STTM Silver to Gold v1.0)'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                          AS year,
    month(test_date)                         AS month,
    CAST(date_trunc('MONTH', test_date) AS DATE) AS month_start,
    COUNT(*)                                 AS total_tests,
    COUNT_IF(result_status = 'Abnormal')     AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')     AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date), CAST(date_trunc('MONTH', test_date) AS DATE)
),
rates AS (
  SELECT
    m.*,
    CAST(abnormal_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
trend AS (
  SELECT
    r.*,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  t.test_name                                                          AS test_name,
  CAST(t.year AS INT)                                                  AS year,
  CAST(t.month AS INT)                                                 AS month,
  -- Calendar attributes via optional join; fall back to derived values when the month is outside the calendar range
  COALESCE(c.month_name, date_format(t.month_start, 'MMMM'))           AS month_name,
  COALESCE(c.quarter_name, concat('Q', quarter(t.month_start)))        AS quarter_name,
  CAST(t.total_tests AS INT)                                           AS total_tests,
  CAST(t.abnormal_test_count AS INT)                                   AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                   AS critical_test_count,
  t.abnormal_rate_pct                                                  AS abnormal_rate_pct,
  t.critical_rate_pct                                                  AS critical_rate_pct,
  CAST(t.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                 AS prev_month_abnormal_rate_pct,
  CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM trend t
LEFT JOIN adb_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(t.month_start, 'yyyyMMdd') AS INT);

In [0]:
updated_df = spark.table("adb_caresync.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count)) 